In [ ]:
gemini_key = ""

In [2]:
!pip install -U langchain-google-genai

In [3]:
import os
os.environ['GOOGLE_API_KEY'] = gemini_key

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [5]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.1)

# Simple text invocation
result = llm.invoke("Sing a ballad of LangChain.")
print(result.content)

# Multimodal invocation with gemini-pro-vision
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "What's in this image?",
        },
        {"type": "image_url", "image_url": "https://picsum.photos/seed/picsum/200/300"},
    ]
)
result = llm.invoke([message])
print(result.content)

(Verse 1)
In realms of code, where knowledge sleeps,
A challenge rose, from digital deeps.
To weave the words, with logic's thread,
And answer questions, long unsaid.
Then LangChain came, a shining light,
To bridge the gap, 'twixt dark and bright.
A framework bold, a promise grand,
To understand, at our command.

(Verse 2)
With LLMs, its heart ablaze,
It parsed the text, in countless ways.
From documents vast, a hidden store,
It sought the truth, and asked for more.
No longer bound by rigid rules,
It learned and grew, like ancient schools.
With chains of thought, it built its quest,
To find the answer, put to the test.

(Verse 3)
From agents wise, to memory deep,
It held the knowledge, secrets to keep.
It spoke with tools, a skillful hand,
To gather data, across the land.
It reasoned well, with logic keen,
A digital mind, yet to be seen.
It navigated complex terrain,
And brought forth answers, clear and plain.

(Verse 4)
But challenges arose, along the way,
The cost of tokens, held the

In [6]:
llm.invoke("What's the capital of india?")

AIMessage(content='The capital of India is **New Delhi**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--e2198118-0033-43b0-93dd-c4e114cddd62-0', usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'cache_read': 0}})

In [7]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables=["Cuisine"],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this. Tell only fency names not any other info"
)

p = prompt_template_name.format(cuisine="italian")
print(p)
result = llm.invoke(p)
print(result.content)

I want to open a restaurant for italian food. Suggest a fency name for this. Tell only fency names not any other info
*   Il Sogno Italiano
*   La Tavola d'Oro
*   Ristorante Paradiso
*   Villa Firenze
*   L'Opera del Gusto
*   Palazzo Italia
*   Il Giardino Segreto
*   La Corona di Roma
*   Antica Osteria
*   Sinfonia Italiana


# Chains


In [8]:
from langchain.chains import LLMChain

chain = LLMChain(llm = llm, prompt = prompt_template_name, verbose = False)
result = chain.invoke("indian")
print(result["text"])

/tmp/ipython-input-3345156815.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm = llm, prompt = prompt_template_name, verbose = False)


*   The Gilded Spices
*   Saffron & Silk
*   The Royal Tandoor
*   Amber Palace
*   The Jewel of Agra
*   Indochine
*   Nirvana Cuisine
*   The Cardamom Club
*   Masala Art
*   Flavors of the Maharajas


In [9]:
prompt_template_name = PromptTemplate(
    input_variables = ["cuisine"],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this. Tell only fency names not any other info"
)

name_chain = LLMChain(llm = llm, prompt = prompt_template_name,  verbose = False)

prompt_template_items = PromptTemplate(
    input_variables = ["restaurant_name"],
    template = "Suggest some menu items for {restaurant_name}. Return them as a comma separated string"
)

food_items_chain = LLMChain(llm = llm, prompt = prompt_template_items, verbose = False)

## Simple Sequential Cahina

In [10]:
from langchain.chains import SimpleSequentialChain
chain = SimpleSequentialChain(chains = [name_chain, food_items_chain])
content = chain.invoke("indian")
print(content)

{'input': 'indian', 'output': "**The Gilded Spices:** Gold Leaf Biryani, Saffron-Infused Korma, Spiced Duck Confit, Gilded Vegetable Samosas, Cardamom & Rosewater Lassi\n\n**Saffron & Silk:** Silk Route Dumplings, Saffron Risotto with Prawns, Kashmiri Lamb Rogan Josh, Apricot & Almond Kofta, Rose Petal Ice Cream\n\n**The Royal Tandoor:** Tandoori Lobster, Murgh Malai Tikka, Seekh Kebab Platter, Tandoori Paneer Tikka, Naan Bread Selection\n\n**Amber Palace:** Amber Chicken Curry, Royal Thali (Vegetarian & Non-Vegetarian), Pistachio & Cardamom Kulfi, Palace Pilaf, Dal Makhani\n\n**The Jewel of Agra:** Agra Fort Chicken Curry, Jewel-Encrusted Naan, Taj Mahal Rice Pudding, Shahi Tukda, Vegetable Jalfrezi\n\n**Indochine:** Vietnamese Spring Rolls, Thai Green Curry, Cambodian Fish Amok, Pad See Ew, Mango Sticky Rice\n\n**Nirvana Cuisine:** Vegetarian Thali, Lentil Soup, Palak Paneer, Vegetable Biryani, Gulab Jamun\n\n**The Cardamom Club:** Cardamom Chicken Curry, Cardamom Infused Lassi, Card

In [11]:
print(content["output"])

**The Gilded Spices:** Gold Leaf Biryani, Saffron-Infused Korma, Spiced Duck Confit, Gilded Vegetable Samosas, Cardamom & Rosewater Lassi

**Saffron & Silk:** Silk Route Dumplings, Saffron Risotto with Prawns, Kashmiri Lamb Rogan Josh, Apricot & Almond Kofta, Rose Petal Ice Cream

**The Royal Tandoor:** Tandoori Lobster, Murgh Malai Tikka, Seekh Kebab Platter, Tandoori Paneer Tikka, Naan Bread Selection

**Amber Palace:** Amber Chicken Curry, Royal Thali (Vegetarian & Non-Vegetarian), Pistachio & Cardamom Kulfi, Palace Pilaf, Dal Makhani

**The Jewel of Agra:** Agra Fort Chicken Curry, Jewel-Encrusted Naan, Taj Mahal Rice Pudding, Shahi Tukda, Vegetable Jalfrezi

**Indochine:** Vietnamese Spring Rolls, Thai Green Curry, Cambodian Fish Amok, Pad See Ew, Mango Sticky Rice

**Nirvana Cuisine:** Vegetarian Thali, Lentil Soup, Palak Paneer, Vegetable Biryani, Gulab Jamun

**The Cardamom Club:** Cardamom Chicken Curry, Cardamom Infused Lassi, Cardamom & Pistachio Cake, Lamb Chops with Cardam

## Sequential Chain

In [12]:
prompt_template_name = PromptTemplate(
    input_variables = ["cuisine"],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this. Tell me only one"
)

name_chain = LLMChain(llm = llm, prompt = prompt_template_name, output_key = "restaurant_name", verbose = False)

prompt_template_items = PromptTemplate(
    input_variables = ["restaurant_name"],
    template = "Suggest some menu items for {restaurant_name}. you should only give items names"
)

food_items_chain = LLMChain(llm = llm, prompt = prompt_template_items, output_key = "menu_items", verbose = False)

In [13]:
from langchain.chains import SequentialChain

chain = SequentialChain(
    chains = [name_chain, food_items_chain],
    input_variables = ["cuisine"],
    output_variables = ["restaurant_name", "menu_items"]
)

In [14]:
result = chain.invoke({'cuisine':"indian"})

In [15]:
print(result["restaurant_name"])

**The Gilded Spices**


In [16]:
print(result["menu_items"])


Okay, here are some menu item names for "The Gilded Spices," aiming for a luxurious, exotic, and flavorful feel:

**Appetizers:**

*   Saffron Kissed Scallops
*   Cardamom & Fig Crostini
*   Turmeric Tempura Vegetables
*   Black Garlic Hummus with Za'atar Pita
*   Chili-Lime Crispy Calamari

**Soups & Salads:**

*   Golden Beet & Ginger Bisque
*   Spiced Lentil & Coconut Soup
*   Pomegranate & Pistachio Salad
*   Cumin-Roasted Carrot Salad

**Main Courses:**

*   Cinnamon-Braised Lamb Shank
*   Star Anise Duck Confit
*   Garam Masala Salmon
*   Smoked Paprika Paella
*   Rosemary & Garlic Filet Mignon
*   Ginger-Glazed Tofu with Forbidden Rice

**Sides:**

*   Cumin Roasted Potatoes
*   Sautéed Spinach with Nutmeg
*   Saffron Risotto
*   Harissa Glazed Carrots

**Desserts:**

*   Cardamom Crème brûlée
*   Rosewater Panna Cotta
*   Chai Spiced Chocolate Cake
*   Pistachio Baklava
*   Gingerbread Ice Cream

**Drinks:**

*   Spiced Pear Martini
*   Saffron Infused Lemonade
*   Cinnamon Old

# Google Search by Langchain

In [ ]:
import os

os.environ["GOOGLE_CSE_ID"] = ""
os.environ["GOOGLE_API_KEY"] = ""

In [18]:
%pip install --upgrade --quiet  langchain-google-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [19]:
from langchain_core.tools import Tool
from langchain_google_community import GoogleSearchAPIWrapper

search = GoogleSearchAPIWrapper()

tool = Tool(
    name="google_search",
    description="Search Google for recent results.",
    func=search.run,
)

In [20]:
print(tool.run("Obama's first name?"))

Child's First Name (Type or print) lb. Middle Name. BARACK. HUSSEIN. CERTIFICATE OF LIVE BIRTH. FILE 151. NUMBER le. DEPARTMENT OF HEALTH. 61. 10641. Last Name. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 ... Jan 28, 2021 ... Obama's name (particularly his middle name Hussein) was the object of xenophobic innuendo questioning his loyalty (especially in the context of ... President Barack Obama, First Lady Michelle Obama, and their daughters, Malia, left, and Sasha, right, sit for a family portrait in the Oval Office, ... Barack Obama ; Kwame Raoul · Barack Hussein Obama II. (1961-08-04) August 4, 1961 (age 64) Honolulu, Hawaii, U.S.. Apr 12, 2017 ... Hussein, Obama's middle name, is a very common Arabic name, meaning "good," "handsome," or "beautiful." Obama's father was a Muslim, and it ... Full name: Barack Hussein Obama Born: 4 August 1961, Honolulu, HI, USA Date ... 

# Agent

In [21]:
!pip install serpapi

In [22]:
!pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=7b604541f63b83a837d8e16f02e943d595860306da1e726cd04729b47dd2c86c
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [23]:
from serpapi import GoogleSearch
print("✅ Import successful")


✅ Import successful


In [24]:
import serpapi

In [ ]:
# make sure yoy have installed this package: pip install google-search-results
# from secret_key import serpapi_key
# os.environ['SERPAPI_API_KEY'] = serpapi_key

os.environ['SERPAPI_API_KEY'] = ""

In [ ]:
from serpapi import GoogleSearch

params = {
  "q": "Coffee",
  "location": "Austin, Texas, United States",
  "hl": "en",
  "gl": "us",
  "google_domain": "google.com",
  "api_key": ""
}

search = GoogleSearch(params)
results = search.get_dict()

In [29]:
results

{'search_metadata': {'id': '68b06167bdfe4955ccd0c896',
  'status': 'Success',
  'json_endpoint': 'https://serpapi.com/searches/4baf025420e4d17f/68b06167bdfe4955ccd0c896.json',
  'pixel_position_endpoint': 'https://serpapi.com/searches/4baf025420e4d17f/68b06167bdfe4955ccd0c896.json_with_pixel_position',
  'created_at': '2025-08-28 14:02:15 UTC',
  'processed_at': '2025-08-28 14:02:15 UTC',
  'google_url': 'https://www.google.com/search?q=Coffee&oq=Coffee&uule=w+CAIQICIaQXVzdGluLFRleGFzLFVuaXRlZCBTdGF0ZXM&hl=en&gl=us&sourceid=chrome&ie=UTF-8',
  'raw_html_file': 'https://serpapi.com/searches/4baf025420e4d17f/68b06167bdfe4955ccd0c896.html',
  'total_time_taken': 2.24},
 'search_parameters': {'engine': 'google',
  'q': 'Coffee',
  'location_requested': 'Austin, Texas, United States',
  'location_used': 'Austin,Texas,United States',
  'google_domain': 'google.com',
  'hl': 'en',
  'gl': 'us',
  'device': 'desktop'},
 'search_information': {'query_displayed': 'Coffee',
  'total_results': 344

In [30]:
from langchain.agents import AgentType, initialize_agent
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_google_genai import ChatGoogleGenerativeAI

In [31]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["serpapi", "llm-math"], llm = llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(tools, llm, agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)


/tmp/ipython-input-3027209921.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(tools, llm, agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)


In [32]:
# Let's test it out!
agent.run("What is the pm of india. tell me name of him. and what is the date of today")

/tmp/ipython-input-857737148.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("What is the pm of india. tell me name of him. and what is the date of today")




> Entering new AgentExecutor chain...
I need to find out who the Prime Minister of India is and today's date.

Action: Search
Action Input: "Prime Minister of India"
Observation: [{'title': 'Modi seeks closer ties on Asia tour to offset fallout from US tariffs', 'link': 'https://www.reuters.com/world/china/modi-seeks-closer-ties-asia-tour-offset-fallout-us-tariffs-2025-08-28/', 'source': 'Reuters', 'source_logo': 'https://serpapi.com/searches/68b061ae6f1e40a0d397df50/images/3bef0655dc9c8c9e6ab58bbe376e23c6b5c0f2308f7fb32a1a01c56b2f547263.png', 'date': '5 hours ago', 'thumbnail': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQC4RLb_e8oN4wqGuMUZyB_vwTU3h8kGaWpykcBGmz6Ee2aMiLobENqZVM&usqp=CAI&s'}, {'title': 'Minister Anand announces the appointment of Canada’s new High Commissioner to India', 'link': 'https://www.canada.ca/en/global-affairs/news/2025/08/minister-anand-announces-the-appointment-of-canadas-new-high-commissioner-to-india.html', 'source': 'Canada.ca', 'source_logo'

"The Prime Minister of India is Narendra Modi. Today's date is August 28, 2025."

## Wikipedia and llm-math tool

In [33]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=3eef1df01056e920692aa006799426675608daf727be229539c126113fa1b9ec
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [35]:
# install this package: pip install wikipedia

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["wikipedia","llm-math"], llm = llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

In [36]:
agent.run("When was Elon musk born? What i shis age right now in 2025")



> Entering new AgentExecutor chain...
I need to find Elon Musk's birthdate and then calculate his age in 2025.
Action: wikipedia
Action Input: Elon Musk birth date
Observation: Page: Musk family
Summary: The Musk family is a prominent family with roots in South Africa, Canada, England, and the United States, whose most famous member is billionaire Elon Musk. It also includes his parents Maye Musk, a model and author, and Errol Musk, a mine-owner, businessman and politician; siblings Kimbal Musk and Tosca Musk, and cousin Lyndon Rive. Elon Musk's ex-wives are Justine Musk and Talulah Riley, and he has many children, including Vivian Wilson. Maye's father Joshua N. Haldeman was a notable chiropractor, aviator, and politician who promoted technocracy and South African apartheid.

Page: Maye Musk
Summary: Maye Musk (née Haldeman; born April 19, 1948) is a model and dietitian. She has been a model for 50 years, appearing on the covers of magazines, including a Time magazine health edition

'Elon Musk was born on June 28, 1971. He will be 54 years old in 2025.'

## Memory

In [37]:
chain = LLMChain(llm = llm, prompt = prompt_template_name)
name = chain.run("maxican")
print(name)

**Alma de México** (Soul of Mexico)


In [38]:
name = chain.run("Indian")
print(name)

**The Gilded Spices**


In [39]:
chain.memory

In [40]:
type(chain.memory)

NoneType

## ConversationBufferMemory

In [41]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

chain = LLMChain(llm = llm, prompt = prompt_template_name, memory = memory)
name = chain.run("Indian")

/tmp/ipython-input-2598159692.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [42]:
print(name)

**The Gilded Spices**


In [43]:
name = chain.run("Arabic")
print(name)

Layali Al Sharq (Nights of the East)


In [44]:
print(chain.memory.buffer)

Human: Indian
AI: **The Gilded Spices**
Human: Arabic
AI: Layali Al Sharq (Nights of the East)


## ConversationChain

In [45]:
from langchain.chains import ConversationChain

convo = ConversationChain(llm = llm)
print(convo.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


/tmp/ipython-input-1483908600.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  convo = ConversationChain(llm = llm)


In [46]:
convo.run("Who won the first cricket world cup?")

"AI: Ah, a question about the gentleman's game! The first Cricket World Cup, officially known as the Prudential Cup (due to sponsorship, you see!), was held in England in 1975. And the winner, after a thrilling final at Lord's, was the **West Indies!** They defeated Australia by 17 runs. Clive Lloyd was the captain of the West Indies team and he also scored a fantastic century in the final, 102 off 85 balls if I recall correctly. A truly memorable moment in cricket history!"

In [47]:
convo.run("How much is 5+5?")

"That's a classic! 5 + 5 equals 10. A fundamental building block of arithmetic, wouldn't you agree? It's amazing how such a simple equation can be the basis for so much more complex mathematics!"

In [49]:
convo.run("Who was the captain ofthe winning team?")

'As I mentioned earlier, the captain of the winning West Indies team in the 1975 Cricket World Cup was the legendary **Clive Lloyd!** He not only captained the team to victory but also played a crucial role in the final, scoring that magnificent century. A true leader and a brilliant player!'

In [48]:
print(convo.memory.buffer)

Human: Who won the first cricket world cup?
AI: AI: Ah, a question about the gentleman's game! The first Cricket World Cup, officially known as the Prudential Cup (due to sponsorship, you see!), was held in England in 1975. And the winner, after a thrilling final at Lord's, was the **West Indies!** They defeated Australia by 17 runs. Clive Lloyd was the captain of the West Indies team and he also scored a fantastic century in the final, 102 off 85 balls if I recall correctly. A truly memorable moment in cricket history!
Human: How much is 5+5?
AI: That's a classic! 5 + 5 equals 10. A fundamental building block of arithmetic, wouldn't you agree? It's amazing how such a simple equation can be the basis for so much more complex mathematics!


## ConversationBufferWindowMemory

In [50]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=1)

convo = ConversationChain(
    llm=llm,
    memory=memory
)
convo.run("Who won the first cricket world cup?")

/tmp/ipython-input-790582290.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=1)


"AI: Ah, a question about the gentleman's game! The first Cricket World Cup, officially known as the Prudential Cup (due to sponsorship, you see!), was held in England in 1975. And the winner, drumroll please... was the **West Indies!** They defeated Australia in the final at Lord's Cricket Ground. A truly memorable moment in cricket history! Do you have any other cricket questions for me? I know a fair bit about the sport, including some of the more obscure rules and historical anecdotes."

In [51]:
convo.run("How much is 5+5?")

"That's a classic! 5 + 5 equals 10. A nice, round number! Is there anything else I can calculate for you? I can handle much more complex equations than that, if you'd like to test my abilities! I can even do some basic statistical analysis or convert between different units of measurement. Just let me know!"

In [52]:
convo.run("Who was the captain of the winning team?")


"Unfortunately, I don't have enough information to answer that question. To tell you who the captain of the winning team was, I'd need to know which game or competition you're referring to. Was it a sports game like football, basketball, or soccer? Or perhaps a different kind of competition, like a debate tournament or a quiz bowl? If you can give me some more details, I'll do my best to find the answer for you! I can access and process information from a vast range of sources, so the more specific you are, the better chance I have of finding the right captain."

In [53]:
print(convo.memory.buffer)

Human: Who was the captain of the winning team?
AI: Unfortunately, I don't have enough information to answer that question. To tell you who the captain of the winning team was, I'd need to know which game or competition you're referring to. Was it a sports game like football, basketball, or soccer? Or perhaps a different kind of competition, like a debate tournament or a quiz bowl? If you can give me some more details, I'll do my best to find the answer for you! I can access and process information from a vast range of sources, so the more specific you are, the better chance I have of finding the right captain.
